# 03. Build the pooled arm from LLM-coded controls

Combines the LLM-coded control severities with the AI arm into `pooled_coded.csv`, which `04`
section E reads. Handles two data issues found during validation:
- duplicate opinions (same text under different captions) -> dedupe on the opinion text
- ineligible cases (criminal community-control, licensing, bar discipline) -> drop `llm_eligible==False`

Inputs (in `../data/coded/`): `controls_all_fulltext.csv`, `controls_llm_coded.csv`, `analysis_data_coded.csv`
Output: `../data/coded/pooled_coded.csv`

In [1]:
import pandas as pd, numpy as np

D = "../data/coded/"
full = pd.read_csv(D+"controls_all_fulltext.csv")      # case_name, year, nature_of_suit, text_for_coding
llm  = pd.read_csv(D+"controls_llm_coded.csv")          # case_name, llm_severity, llm_eligible, ...

# 1) collapse duplicate opinions (identical text under different captions)
full = full.drop_duplicates(subset=["text_for_coding"]).copy()
# 2) attach LLM codes (dedupe llm on case_name first), keep eligible only
llm = llm.drop_duplicates("case_name", keep="first")
ctrl = full.merge(llm[["case_name","llm_severity","llm_eligible"]], on="case_name", how="left")
ctrl = ctrl[ctrl["llm_eligible"] == True].copy()
ctrl = ctrl.dropna(subset=["llm_severity"])
print(f"controls after dedupe + eligible filter: {len(ctrl)}")

# 3) map nature_of_suit -> field (coarse)
def field_from_nos(nos):
    if not isinstance(nos,str): return "other"
    s=nos.lower()
    for k,v in [("civil right","civil rights"),("contract","contract"),("tort","tort"),
                ("employ","employment"),("labor","employment"),("administrativ","administrative"),
                ("family","family")]:
        if k in s: return v
    return "other"

controls = pd.DataFrame({
    "case_name": ctrl["case_name"], "year": ctrl.get("year"), "ai": 0,
    "severity": ctrl["llm_severity"].astype(int),
    "pro_se": np.nan, "federal": np.nan,
    "field": ctrl["nature_of_suit"].map(field_from_nos)})

controls after dedupe + eligible filter: 321


In [2]:
# 4) AI arm
ai = pd.read_csv(D+"analysis_data_coded.csv")
ai_arm = pd.DataFrame({
    "case_name": ai.get("Case Name"), "year": ai.get("year"), "ai": 1,
    "severity": ai["severity"].astype(int), "pro_se": ai.get("pro_se"),
    "federal": ai.get("federal"), "field": ai.get("field")})

pooled = pd.concat([ai_arm, controls], ignore_index=True)
pooled.to_csv(D+"pooled_coded.csv", index=False)
print(f"wrote {D}pooled_coded.csv  (AI={int((pooled.ai==1).sum())}, control={int((pooled.ai==0).sum())})")
print("\nseverity by arm:")
print(pooled.groupby("ai")["severity"].value_counts().unstack().fillna(0).astype(int).to_string())
print("\nmean severity by arm:", pooled.groupby("ai")["severity"].mean().round(2).to_dict())
print("\nNow run 04 (analyze) for the AI figures + pooled regression.")

wrote ../data/coded/pooled_coded.csv  (AI=1279, control=321)

severity by arm:
severity    0    1    2    3    4
ai                               
0         203   15   20   28   55
1         256  498  217  135  173

mean severity by arm: {0: 1.12, 1: 1.59}

Now run 04 (analyze) for the AI figures + pooled regression.
